In [ ]:
import os
import sys; sys.path.append(os.path.join(os.path.abspath(''), '../experiments'));
sys.path.append("../../")

import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

import json


In [ ]:
import periodic_simulation_setup

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False
stiffness_pressure = 0.4
scale_factor_pressure = 0.01
avg_len = 0.1

In [ ]:
tag = '0.2_1.4_30.00'
amp = float(tag.split('_')[0])
r = float(tag.split('_')[1])
angle = float(tag.split('_')[2])

In [ ]:


h = 5
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])
pts, edges, bpts, bedges = pattern_generator_using_gmsh.get_cosine_dash(h, avg_len, avg_len, amplitude=amp, dash_point = dash_point, return_line_segments=True)
new_points = list(pts) + list(bpts)
new_edges = list(edges) + list(bedges + len(pts))
visualization.plot_line_segments(new_points, np.array(new_edges) - 1)

In [ ]:
%%capture

h = 5
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])
ipu, m, marker = pattern_generator_using_gmsh.get_cosine_dash(h, avg_len, avg_len, amplitude=amp, dash_point = dash_point)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
configure_solver_parallelism()

In [ ]:
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
hessianShiftForRigidMotion = 1e-10
hessianShiftForAlphainPlanar = 1e-12

In [ ]:
fixedVars, hessianShift = [], hessianShiftForRigidMotion

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

In [ ]:
opts.niter = 500
opts.gradTol = 1e-10

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

In [ ]:
fixedVars, hessianShift = list(ipu.get_center_fixedVars()), hessianShiftForAlphainPlanar

In [ ]:
opts.niter = 500
opts.gradTol = 1e-10

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

In [ ]:
experiment_log = {}
experiment_log["Ipu simulation succeed"] = int(cr.success)
experiment_log["Simulation Kappa value"] = (ipu.getVars()[-2])
if np.abs(ipu.getVars()[-2]) > 1e-10:
    experiment_log["Planar equilibrium"] = 0
    print("Warning: Can not compute stiffness due to non-planar equilibrium!")
else:
    experiment_log["Planar equilibrium"] = 1

In [ ]:
result_folder = 'output'
name = 'cosine_dash'
variable = 0
render_images = True

In [ ]:
# Compute stiffness after removing the vertical offset
ipu.reparametrize_vertical_offset()
optimizer = inflation.get_inflation_optimizer(ipu, ipu.getBendingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)
stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(ipu, 1000, optimizer, hessianShift = 0, fixedVars = ipu.getBendingStiffnessFixedVars(), filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable), generate_images = render_images)

In [ ]:
from IPython.display import Image
Image(filename="{}/stiffness_{}_{}.png".format(result_folder, name, variable)) 

In [ ]:
min(stiffness_values), max(stiffness_values)

In [ ]:
ipu.get_deformation_scale_factors()

In [ ]:
np.set_printoptions(suppress=True, precision=6)

In [ ]:
betas = np.linspace(0, 2 * np.pi, 1000)

In [ ]:
optimizer = inflation.get_inflation_optimizer(ipu, ipu.getStretchingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)
stretchingStiffness = inflation.getStretchingStiffness(ipu, betas, optimizer, 0, ipu.getStretchingStiffnessFixedVars())

In [ ]:
plot_min_r = 0
plot_max_r = None
r = list(stretchingStiffness)
# + list(stretchingStiffness)
# theta = list(sampled_alpha) + list(np.pi + np.array(sampled_alpha))
theta = list(betas)

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot(theta, r)
ax.set_rmax(max(stretchingStiffness) if plot_max_r is None else plot_max_r)
ax.set_rmin(min(stretchingStiffness) - 0.2 * (max(stretchingStiffness) - min(stretchingStiffness)) if plot_min_r is None else plot_min_r)
# ax.set_rticks([0.5, 1, 1.5, 2])  # Less radial ticks
# ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
ax.grid(True)

ax.set_title("Stretching stiffness", va='bottom')
plt.tight_layout()
plt.savefig("{}/stretching_stiffness_{}_{}.png".format(result_folder, name, 49), dpi = 300) 

In [ ]:
np.min(stretchingStiffness), np.max(stretchingStiffness)

In [ ]:
import periodic_simulation_setup

In [ ]:
points = periodic_simulation_setup.visualize_average_deformation_gradient(ipu, 100, plot_max_r=1, plot_min_r=0, show_figure=True, filename = "{}/average_deformation_gradient_{}_{}.png".format(result_folder, name, variable))